# V3-A 实时50 Hz静止基准
先停止全部旧监控与采集内核，在独立heart_v3a_live目录使用本Notebook。不要Run All。不要覆盖旧目录。
本次仍为固定三轴限幅NLMS＋冻结PS规则，不含摄像头。初次只采60秒静止基准，界面同时刷新；不是正式验收。

In [ ]:
# ① 身份检查：不加载硬件。Notebook应位于解压后的新包根目录。
import sys, pathlib, json
import pynq
print('目录：', pathlib.Path.cwd())
print('Python：', sys.executable, 'PYNQ：', pynq.__version__)
!{sys.executable} run_v3a_live.py --check-bundle
!{sys.executable} run_v3a_board_replay.py --check-bundle
from heart_v3a.live.board import check_lock_exclusion
print('两进程锁检查：', check_lock_exclusion())


In [ ]:
# ② 显式启动：先放稳手指与IMU，保持接触压力，不要拉扯连线。
import time, uuid
from datetime import datetime
from heart_v3a.live.board import prepare_live_session
from heart_v3a.live.contracts import LiveSettings
from heart_v3a.live.view import NotebookView
if globals().get('session') is not None and session.snapshot().state not in ('stopped', 'faulted'):
    raise RuntimeError('请先运行停止单元并确认cleanup_complete，不要重新启动')
bundle = pathlib.Path.cwd()
output = bundle / 'live_results' / ('baseline-' + datetime.now().strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8])
settings = LiveSettings(mode='baseline', protocol='static', duration_seconds=60.0)
print('60秒静止基准；启动后立即运行下一查看单元，界面需要同时刷新。')
for remaining in range(5, 0, -1):
    print('准备倒计时：', remaining)
    time.sleep(1)
session = None
view = None
try:
    session = prepare_live_session(bundle, output, settings, hardware=True)
    view = NotebookView()
    session.start()
except BaseException:
    if session is not None:
        print(session.stop(display_metrics=view.metrics() if view is not None else None))
    else:
        print('预检未完成，没有建立采集记录。请保留上方错误，不要继续运行查看单元。')
    raise
print('已启动，请立即运行③。')


In [ ]:
# ③ 有限查看：每0.5秒读内存快照。不卡采集线程。自然结束后显式清理。
if globals().get('session') is None or globals().get('view') is None:
    raise RuntimeError('没有已就绪会话／显示，请先处理②中的错误')
try:
    while session.snapshot().state == 'running':
        view.refresh(session.snapshot())
        time.sleep(0.5)
except KeyboardInterrupt:
    print('查看已中断，正在请求停止；底层调用未退出时不会宣称已清理。')
finally:
    try:
        view.finish(session.snapshot())  # 结束查看先清N/A，不计为实时延迟样本。
    finally:
        summary = session.stop(display_metrics=view.metrics())
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('结果目录：', session.paths.directory if session.paths else '尚无文件')


In [ ]:
# ④ 停止／汇总：可重复运行；若停止未完成，勿重开会话或覆盖文件。
if globals().get('session') is None:
    print('尚无采集会话，请保留预检错误；没有结果文件。')
else:
    try:
        if globals().get('view') is not None:
            view.finish(session.snapshot())
    finally:
        summary = session.stop(display_metrics=view.metrics() if globals().get('view') is not None else None)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print('四份文件：', session.paths)
    if not summary.get('cleanup_complete'):
        print('停止未完成：保留硬件所有权，等待后再运行本停止单元；不要运行启动单元。')
    elif summary.get('fault'):
        print('本次故障：保存四份结果，排查连线／错误，不自动重试或恢复。')


In [ ]:
# ⑤ 下载真实文件：点下方四个链接保存并交回，不要只交summary。
from IPython.display import display, FileLink
if globals().get('session') is None or session.paths is None:
    print('未建立记录目录，无法下载。')
else:
    for path in (session.paths.raw_csv, session.paths.results_csv, session.paths.events_jsonl, session.paths.summary_json):
        if path.is_file():
            display(FileLink(str(path.relative_to(bundle))))
        else:
            print('文件未生成：', path.name, '请同时交回停止摘要。')


## 后续停点
交回raw.csv、results.csv、events.jsonl、summary.json后，先审计基准并整体确认记录率／观察间隔／延迟限值；本包没有预填approved限值。
静止、20秒静止→20秒轻晃→20秒静止、20秒正常接触→10秒离开→30秒恢复及两次启停，须在批准限值后依次正式验收。当前live／DDR／自动beta效果／V3-A完整门均未通过。